# 05 — DTC_breeze + Yol Yoğunluğu

İki bağımsız değişken üretilir:

**1. DTC_breeze (Distance To Coast — Breeze direction)**  
Hakim rüzgar 165° (SSE) yönünden geliyor. Her grid hücresinden bu yönde
ışın çekilir; ışının ilk OSM kıyı çizgisini kestiği noktaya olan mesafe
= rüzgar-yönelimli kıyı mesafesi. Düşük değer = serin deniz havası bu
hücreye kolay ulaşır; yüksek değer = bina/kara katmanı arada.

**2. Yol yoğunluğu (`road_density_m_per_km2`)**  
OSM araç yolları (`network_type='drive'`) her 30 m hücre içine
kesilip toplam uzunluk hesaplanır; alan başına normalize edilir.

**Önkoşul:** `grid_30m_building.gpkg` (Hafta 5).

**Çıktı:** `grid_30m_full.gpkg` — **7 bağımsız değişkenin tamamı + LST**.
Hafta 8 standardize, Hafta 9 VIF için hazır.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_GRID, DATA_PROCESSED, FIGURES, TABLES,
    CRS_PROJECTED, WIND_FROM_DEG, DTC_MAX_DISTANCE_M,
    GRID_30M_BUILDING, GRID_30M_FULL,
    COASTLINE_GPKG, ROADS_GPKG,
)
from src.dtc_breeze import (
    load_osm_coastline, load_osm_roads,
    dtc_breeze_for_geometries, road_length_per_cell,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Pilot sınır ve grid ---
boundary = gpd.read_file(DATA_GRID / "pilot_boundary.gpkg")
grid = gpd.read_file(GRID_30M_BUILDING)
print(f"Pilot sınır CRS: {boundary.crs}")
print(f"Grid: {len(grid):,} hücre, kolonlar: {[c for c in grid.columns if c != 'geometry']}")

In [ ]:
# --- OSM coastline (cache'li) ---
if COASTLINE_GPKG.exists():
    coastline = gpd.read_file(COASTLINE_GPKG)
    print(f"Cache'den yüklendi: {COASTLINE_GPKG.name} ({len(coastline)} parça)")
else:
    print("OSM'den çekiliyor (ilk seferde 30s-2dk sürebilir)...")
    coastline = load_osm_coastline(boundary, buffer_m=5000)
    coastline.to_file(COASTLINE_GPKG, driver="GPKG")
    print(f"Kaydedildi: {COASTLINE_GPKG.name}")

total_len_km = coastline.geometry.length.sum() / 1000
print(f"\nCoastline toplam uzunluk: {total_len_km:.2f} km")
print(f"Geometri tipleri: {coastline.geometry.geom_type.value_counts().to_dict()}")

In [ ]:
# --- DTC_breeze ray casting ---
import time
t0 = time.time()
print(f"Ray casting: {len(grid):,} hücre × açı={WIND_FROM_DEG}° SSE...")

dtc = dtc_breeze_for_geometries(
    geometries=grid.geometry,
    coastline=coastline,
    wind_from_deg=WIND_FROM_DEG,
    max_dist_m=DTC_MAX_DISTANCE_M,
)
grid["dtc_breeze_m"] = dtc

print(f"Bitti ({time.time()-t0:.1f}s)")
print(f"\nDTC_breeze özet:")
print(pd.Series(dtc).describe().round(1))

saturated = (dtc >= DTC_MAX_DISTANCE_M * 0.999).sum()
if saturated > 0:
    print(f"\n⚠ {saturated:,} hücre ({100*saturated/len(grid):.1f}%) max_dist'te saturated")

In [ ]:
# --- OSM yol ağı ---
if ROADS_GPKG.exists():
    roads = gpd.read_file(ROADS_GPKG)
    print(f"Cache'den yüklendi: {ROADS_GPKG.name} ({len(roads):,} edge)")
else:
    print("OSM'den çekiliyor (graph_from_polygon, 30s-2dk)...")
    roads = load_osm_roads(boundary, network_type="drive", buffer_m=500)
    # GPKG yazımı için MultiIndex'i sıfırla ve sadece geometri tut
    roads = roads.reset_index()[["geometry"]]
    roads.to_file(ROADS_GPKG, driver="GPKG")
    print(f"Kaydedildi: {ROADS_GPKG.name}")

total_road_km = roads.geometry.length.sum() / 1000
print(f"\nToplam yol uzunluğu: {total_road_km:.1f} km")

In [ ]:
# --- Hücre başına yol uzunluğu (overlay) ---
t0 = time.time()
print("Yollar grid hücreleriyle kesiliyor (overlay)...")
grid = road_length_per_cell(roads, grid, cell_id_col="cell_id")
print(f"Bitti ({time.time()-t0:.1f}s)")

n_with_road = (grid["road_length_m"] > 0).sum()
print(f"\nYol içeren hücre: {n_with_road:,} / {len(grid):,} (%{100*n_with_road/len(grid):.1f})")
print(f"\nroad_density_m_per_km2 özet:")
print(grid["road_density_m_per_km2"].describe().round(1))

In [ ]:
# --- 7 değişkenli korelasyon matrisi ---
feat_cols = [
    "lst_mean",
    "ndvi_mean", "albedo_mean", "impervious_pct",
    "building_height_mean", "building_density_per_km2",
    "road_density_m_per_km2", "dtc_breeze_m",
]
available = [c for c in feat_cols if c in grid.columns]
print(f"Mevcut değişkenler: {available}")

corr = grid[available].corr().round(3)
print("\nKorelasyon matrisi (Pearson):")
print(corr)

In [ ]:
# --- Görselleştirme: 4 panel + korelasyon ---
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
specs = [
    ("lst_mean",                "LST ortalama (°C)",          "inferno"),
    ("dtc_breeze_m",            "DTC_breeze (m, 165° SSE)",   "viridis_r"),
    ("road_density_m_per_km2",  "Yol yoğunluğu (m/km²)",      "OrRd"),
    ("building_density_per_km2","Yapı yoğunluğu (b/km²)",     "OrRd"),
    ("impervious_pct",          "Geçirimsiz oran (%)",        "OrRd"),
]
for ax, (col, title, cmap) in zip(axes.ravel()[:5], specs):
    if col not in grid.columns:
        ax.set_title(f"{title} (kolon yok)"); ax.axis("off"); continue
    vals = grid[col].dropna()
    grid.plot(
        column=col, cmap=cmap, ax=ax,
        legend=True, legend_kwds={"label": title, "shrink": 0.6},
        linewidth=0,
        vmin=np.percentile(vals, 2) if len(vals) > 0 else 0,
        vmax=np.percentile(vals, 98) if len(vals) > 0 else 1,
        missing_kwds={"color": "#eaeaea"},
    )
    boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color="#0a9396", linewidth=0.8)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.ticklabel_format(style="plain")

# 6. panel — korelasyon ısı haritası
ax = axes.ravel()[5]
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, cbar_kws={"label": "Pearson r"}, vmin=-1, vmax=1,
            annot_kws={"size": 7})
ax.set_title("7 değişken + LST korelasyon")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "05_dtc_road_overview.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- 7 değişkenli son tabloyu kaydet ---
grid.to_file(GRID_30M_FULL, driver="GPKG")
print(f"Kaydedildi: {GRID_30M_FULL} ({GRID_30M_FULL.stat().st_size/1024/1024:.2f} MB)")

# Summary CSV
lst_corrs = corr["lst_mean"].drop("lst_mean")
summary = pd.DataFrame({
    "degisken": lst_corrs.index,
    "r_with_LST": lst_corrs.values.round(3),
    "min":    [grid[c].min() for c in lst_corrs.index],
    "max":    [grid[c].max() for c in lst_corrs.index],
    "medyan": [grid[c].median() for c in lst_corrs.index],
}).round(3)
summary.to_csv(TABLES / "05_dtc_road_summary.csv", index=False, encoding="utf-8-sig")
print("\nÖzet:")
print(summary.to_string(index=False))

## Özet

- **DTC_breeze** ray casting (165° SSE, Antalya yaz hakim rüzgarı; ERA5 ile hâlâ doğrulanması gerekecek) ile her hücre için hesaplandı.
- **OSM yol ağı** (`network_type='drive'`) çekildi, hücrelerle kesilip toplam uzunluk → yoğunluk türetildi.
- 30 m grid'de **7 bağımsız değişken + LST** artık tek tabloda: `data/processed/grid_30m_full.gpkg`.

## Sınırlılıklar (tezde)

- Rüzgar yönü 165° **literatür/iklim atlas**ından alındı, ERA5 saatlik veriyle doğrulanmadı (sonraki adım).
- Ray casting tek-yön; meltem **bant** etkisi (15° genişlik) için ortalama almak daha sağlam olurdu.
- OSM yol ağı sadece `drive` — yaya yolları/kaldırımlar termal etkide farklı, ayrı bir kolon olarak eklenebilir.
- `graph_from_polygon` simplified=True; küçük detaylar (cul-de-sac) kayıp olabilir.

## Sonraki Adım

`06_standardization.ipynb` — z-skor + log dönüşümler. Sonra `07_multicollinearity` (VIF: 7 → 5 değişken seçimi).